# Frozen-Split Validation Viewer — B0 / B1  ·  Gate 1 Results

**Purpose:** one-click stop to show the professor what the frozen split *means* and how the two baselines behave on *exactly* the same data.  
Uses the official **70/10/5/15 patient-level split** in `data/manifests/split_v1.json` (hash beside it) — no peeking at test, no re-splitting.  

> **How to demo (2 min):** run all cells top-to-bottom.  Cells 1–5 are **fixed** (locked validation contract: `discover`, `model`, `load_case`, `sliding_window_logits`, `regions`, `dice_empty_rule`).  Cells 6+ are the *research pipeline* — qualitative overlays and quantitative tables the professor can read in one glance.  Toggle `CONFIG["model"]` between `"b0"` and `"b1"` to compare the architecture-controlled baselines without changing anything else.

- **B0** = `SegResNetB0` (compact SegResNet, the reference).  
- **B1** = `B1Segmentor` from `cnn_shared.py` — same `SharedStem → CNNEncoder → UNetDecoder` that **H0 will reuse** and extend with ViT+fusion.  That is the whole point of B1: the *only* difference between B1 and H0 will be the mechanism under test.
- **Training harness:** both models train with the *same* `scripts/train_segmentation.py` (same frozen split, seed 17, 150 epochs, AdamW, patch sampler, Dice+CE).  

Outputs saved: `notebooks/frozen_validation_viewer_slice_*.png` (qualitative), `per_case.csv`/`summary.json` (quantitative), all reproducible from the manifest hash.

## 0 — Setup & reproducibility (edit only the paths here)
Fixed budget: 150 epochs, seed 17, patch 96³/stride 48.  Both models share the hardware profile (`allocated_gb` *and* `reserved_gb`) written before training — if a step had fit only on `allocated` the Gate-2 number would not have been comparable.

In [ ]:
import sys, json, csv
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib import colormaps as mpl_cmaps
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "src" / "scanvidence").is_dir() or (p / "scanvidence").is_dir():
            return p
    return start.parent

project_root = find_repo_root()
for cand in (project_root / "src", project_root):
    sys.path.insert(0, str(cand))

CONFIG = {
    # ---- toggle for the professor: "b0" (SegResNetB0) or "b1" (B1Segmentor / cnn_shared) ----
    "model": "b0",
    # frozen split (single source of truth)
    "split_json": "data/manifests/split_v1.json",
    # training data root (BraTS-GLI TrainingData folder)
    "training_data_root": r"C:\Users\NCC-HPC8\Desktop\BraTSGLI\data\ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData\ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    # checkpoints — frozen B0 is the Gate-2 term; fallback to pre-freeze if retrain not yet on disk
    "ckpt_b0": "runs/full-b0-seed17-frozen/best.pt",
    "ckpt_b0_fallback": "runs/full-b0-seed17/best.pt",
    "ckpt_b1": "runs/b1-seed17/best.pt",
    "widths": (16, 32, 64, 128), "num_classes": 4,
    "patch": 96, "stride": 48,
    "max_cases": 20,   # None = every frozen-val case (125); 20 is 1-min demo on T1000
    "expected_params_b0": 1599420,
    "seed": 17,
}
# resolve checkpoint for the selected model
def _resolve_ckpt(cfg):
    if cfg["model"] == "b1":
        return cfg["ckpt_b1"]
    p = project_root / cfg["ckpt_b0"]
    return cfg["ckpt_b0"] if p.exists() else cfg["ckpt_b0_fallback"]
CONFIG["checkpoint_path"] = _resolve_ckpt(CONFIG)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"repo root: {project_root} | device: {device} | model: {CONFIG['model']} -> {CONFIG['checkpoint_path']}")
sha = (project_root / (CONFIG["split_json"] + ".sha256"))
print(f"split: {CONFIG['split_json']}  hash: {sha.read_text().strip()[:12] + '…' if sha.exists() else 'MISSING — run freeze_split.py'}")

## 1 — `discover`  ·  FIXED (locked validation contract)
Do **not** edit. Finds every `<case>-<mod>.nii.gz` by suffix, groups by case, handles inner-folder layout and `BraTSDataset` fallback.  Incomplete cases (missing any of t1n/t1c/t2w/t2f) are dropped here; nothing re-splits later.

In [ ]:
MOD_KEYS = ("seg", "t1n", "t1c", "t2w", "t2f")

def _modality_of(name):
    low = name.lower()
    for key in MOD_KEYS:
        for sep in ("-", "_"):
            for ext in (".nii.gz", ".nii"):
                if low.endswith(f"{sep}{key}{ext}"):
                    return key
    return None

def rglob_discover(root):
    groups = defaultdict(dict)
    for f in root.rglob("*"):
        if not f.is_file(): continue
        k = _modality_of(f.name)
        if not k: continue
        cid = f.name
        for k2 in MOD_KEYS:
            for sep in ("-", "_"):
                for ext in (".nii.gz", ".nii"):
                    suf = f"{sep}{k2}{ext}"
                    if cid.lower().endswith(suf): cid = cid[:-len(suf)]
        groups[cid][k] = f
    recs = []
    for cid, fm in sorted(groups.items()):
        recs.append({"patient_id": cid, **fm,
                     "seg_path": fm.get("seg"),
                     "available_sequences": [k for k in fm if k != "seg"]})
    return recs

def discover(root):
    root = Path(root)
    recs = rglob_discover(root)
    if not recs and (root / root.name).is_dir():      # duplicated-inner-folder layout
        recs = rglob_discover(root / root.name)
    if not recs:                                       # try repo dataset class last
        try:
            from scanvidence.data.datasets import BraTSDataset
            recs = BraTSDataset(str(root), track="GLI").discover()
        except Exception as e:
            print("BraTSDataset fallback failed:", e)
    needed = {"t1n", "t1c", "t2w", "t2f"}
    recs = [r for r in recs if needed <= set(r["available_sequences"])]
    return recs

# --- frozen split is the ONLY allowed validation source ---
split = json.loads((project_root / CONFIG["split_json"]).read_text())
print(f"frozen split: train {len(split['train'])} | val {len(split['val'])} | cal {len(split['calibration'])} | test {len(split['test'])} (total {len(split['train'])+len(split['val'])+len(split['calibration'])+len(split['test'])})")
avail_ids = {r["patient_id"] for r in discover(CONFIG["training_data_root"])}
missing = [c for c in split["val"] if c not in avail_ids]
print(f"frozen-val present on disk: {len(split['val'])-len(missing)}/{len(split['val'])}" + (f"  MISSING e.g. {missing[:3]}" if missing else ""))
# build the validation record list from the FROZEN ids (patient-level, same as training)
all_recs = {r["patient_id"]: r for r in discover(CONFIG["training_data_root"])}
cases = [all_recs[cid] for cid in split["val"] if cid in all_recs]
if CONFIG["max_cases"]: cases = cases[:CONFIG["max_cases"]]
print(f"using {len(cases)} frozen-val cases for this run (set CONFIG['max_cases']=None for all 125)")

## 2 — `model`  ·  FIXED (add B1 branch without touching B0)
Architecture-controlled: B0 uses `SegResNetB0`, B1 uses `B1Segmentor` from `cnn_shared.py` (same `SharedStem → CNNEncoder → UNetDecoder` H0 will later extend). Checkpoint shape, budget and seed are identical — only the architecture changes.

In [ ]:
ckpt_path = project_root / CONFIG["checkpoint_path"]
if not ckpt_path.exists():
    raise FileNotFoundError(f"checkpoint missing: {ckpt_path} — train it with scripts/train_segmentation.py --model {CONFIG['model']}")

if CONFIG["model"] == "b0":
    from scanvidence.models.backbone import SegResNetB0
    model = SegResNetB0(in_channels=4, num_classes=CONFIG["num_classes"], widths=CONFIG["widths"], dropout=0.0)
else:
    from scanvidence.models.cnn_shared import B1Segmentor
    model = B1Segmentor(in_channels=4, num_classes=CONFIG["num_classes"], widths=CONFIG["widths"], dropout=0.0)

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
state = ckpt.get("state_dict", ckpt.get("model", ckpt)) if isinstance(ckpt, dict) else ckpt
model.load_state_dict(state)
n_params = sum(p.numel() for p in model.parameters())
print(f"{CONFIG['model'].upper()} params {n_params} | epoch {ckpt.get('epoch', '?')} | best_val_dice {ckpt.get('best_val_dice', '?'):.4f} | split_hash {str(ckpt.get('split_hash',''))[:10]}…")
if CONFIG["model"] == "b0":
    assert n_params == CONFIG["expected_params_b0"], f"B0 param mismatch {n_params} vs {CONFIG['expected_params_b0']}"
model.eval().to(device)

## 3 — `load_case` + `sliding_window_logits`
Normalization and sliding-window aggregation are part of the locked evaluation contract.  `sliding_window_logits` averages logits voxel-wise across overlapping 96³ windows (stride 48) and crops back — weighted/blended variants would be a different number.

In [ ]:
def normalize_modality(v):                      # MUST match training: z-score on nonzero
    v = np.asarray(v, dtype=np.float32)
    m = v > 0
    out = np.zeros_like(v)
    if m.sum(): out[m] = (v[m] - v[m].mean()) / (v[m].std() + 1e-8)
    return out

def mod_path(rec, key):
    for cand in (rec.get(key), (rec.get("paths") or {}).get(key)):
        if cand: return Path(cand)
    raise KeyError(f"{key} missing for {rec['patient_id']}")

def load_case(rec):
    vol = np.stack([normalize_modality(nib.load(str(mod_path(rec, k))).get_fdata())
                    for k in ("t1n", "t1c", "t2w", "t2f")]).astype(np.float32)
    seg = nib.load(str(rec["seg_path"])).get_fdata().astype(np.int16) if rec.get("seg_path") else None
    if seg is not None:
        vals = set(np.unique(seg).tolist())
        assert vals <= {0, 1, 2, 3}, f"label drift {vals} — STOP"
    return vol, seg

def sliding_window_logits(model, vol, patch, stride, device):
    orig = vol.shape[1:]
    pad = tuple((patch - s % patch) % patch for s in orig)
    if any(pad):
        vol = np.pad(vol, ((0, 0), (0, pad[0]), (0, pad[1]), (0, pad[2])))
    _, D, H, W = vol.shape
    def starts(n):
        s = list(range(0, n - patch + 1, stride))
        if s[-1] != n - patch: s.append(n - patch)
        return s
    acc = np.zeros((4, D, H, W), np.float64); cnt = np.zeros((D, H, W), np.float64)
    with torch.no_grad():
        for i in starts(D):
            for j in starts(H):
                for k in starts(W):
                    x = torch.from_numpy(vol[:, i:i+patch, j:j+patch, k:k+patch])[None].to(device)
                    out = model(x)
                    if isinstance(out, (tuple, list)): out = out[0]
                    acc[:, i:i+patch, j:j+patch, k:k+patch] += out[0].cpu().numpy()
                    cnt[i:i+patch, j:j+patch, k:k+patch] += 1
    return (acc / cnt[None])[(slice(4),) + tuple(slice(0, o) for o in orig)]

## 4 — `regions` + `dice_empty_rule`  ·  FIXED
BraTS convention: empty GT & empty prediction → 1.0 (not 0), otherwise standard Dice.  This is the number the Gate-2 promotion rule will use.

In [ ]:
def regions(L):
    return {"ET": L == 3, "TC": (L == 1) | (L == 3), "WT": (L >= 1) & (L <= 3)}

def dice_empty_rule(a, b):
    a, b = a.astype(bool), b.astype(bool)
    if not a.any() and not b.any(): return 1.0
    if not a.any() or not b.any(): return 0.0
    return 2.0 * (a & b).sum() / (a.sum() + b.sum())

## 5 — Smoke check: one case, three views (qualitative)
Picks the axial slice with the most tumor, renders FLAIR → GT → prediction.  Colors are fixed (0 bg, 1 NCR, 2 ED, 3 ET) so the professor can read them across cases.  Also writes `b0_validation_slice.png` for the slide deck.

In [ ]:
rec = cases[0]
vol, seg = load_case(rec)
pred = sliding_window_logits(model, vol, CONFIG["patch"], CONFIG["stride"], device).argmax(0).astype(np.int16)
print(f"{rec['patient_id']}: predicted labels {np.unique(pred)}")
if seg is not None:
    gt_r, pd_r = regions(seg), regions(pred)
    for r in ("ET", "TC", "WT"):
        print(f"  {r} Dice {dice_empty_rule(pd_r[r], gt_r[r]):.4f}  (constant predictor: {1.0 if gt_r[r].sum()==0 else 0.0:.1f})")
    # pick slice with largest tumor
    sums = seg.sum(axis=(1, 2)); si = int(np.argmax(sums)) if sums.max() > 0 else seg.shape[0] // 2
    cmap = mpl_cmaps["tab10"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(vol[3, si], cmap="gray"); axes[0].set_title(f"FLAIR  {rec['patient_id']}  z={si}"); axes[0].axis("off")
    axes[1].imshow(seg[si], cmap=cmap, vmin=0, vmax=3); axes[1].set_title("Ground truth"); axes[1].axis("off")
    axes[2].imshow(pred[si], cmap=cmap, vmin=0, vmax=3); axes[2].set_title(f"{CONFIG['model'].upper()} pred"); axes[2].axis("off")
    out_dir = project_root / "notebooks"; out_dir.mkdir(exist_ok=True)
    plt.tight_layout(); plt.savefig(out_dir / "b0_validation_slice.png", dpi=150, bbox_inches="tight"); plt.show()

## 6 — Frozen-val quantitative sweep (all cases in this run)
Runs the same contract over every selected frozen-val case, writes `per_case.csv` + `b0_validation_summary.json`.  The constant-predictor column exposes how much the frozen val is inflated by empty ETs.

In [ ]:
rows = []
for rec in cases:
    vol, seg = load_case(rec)
    if seg is None: continue
    pred = sliding_window_logits(model, vol, CONFIG["patch"], CONFIG["stride"], device).argmax(0).astype(np.int16)
    gt_r, pd_r = regions(seg), regions(pred)
    row = {"case": rec["patient_id"]}
    for r in ("ET", "TC", "WT"):
        row[f"dice_{r}"] = round(dice_empty_rule(pd_r[r], gt_r[r]), 4)
        row[f"const_{r}"] = 1.0 if gt_r[r].sum() == 0 else 0.0
    row["acc"] = round(float((pred == seg).mean()), 4)
    rows.append(row)
    print(f"{row['case']}: ET {row['dice_ET']:.3f} TC {row['dice_TC']:.3f} WT {row['dice_WT']:.3f}")
out_dir = project_root / "notebooks"; out_dir.mkdir(exist_ok=True)
with open(out_dir / "per_case.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
summary = {"model": f"{CONFIG['model'].upper()} ({'B1Segmentor' if CONFIG['model']=='b1' else 'SegResNetB0'})", "checkpoint": CONFIG["checkpoint_path"], "n_cases": len(rows), "device": str(device), "seed": CONFIG["seed"], "split": CONFIG["split_json"], "split_hash": sha.read_text().strip() if sha.exists() else ""}
for r in ("ET", "TC", "WT"):
    vals = [x[f"dice_{r}"] for x in rows]
    const = [x[f"const_{r}"] for x in rows]
    summary[r] = {"mean": round(float(np.mean(vals)), 4), "std": round(float(np.std(vals)), 4), "min": round(float(np.min(vals)), 4), "max": round(float(np.max(vals)), 4), "constant_mean": round(float(np.mean(const)), 4)}
(out_dir / "b0_validation_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 7  Research pipeline: prediction vs ground truth

High-readability overlays on the three most informative axial slices per case.  
- **Row 1** T1c (anatomy), **Row 2** FLAIR, **Row 3** GT (opaque), **Row 4** prediction, **Row 5** FLAIR + translucent GT·Pred error map (green = agreement, red = FP, blue = FN).  
Four cases: **best**, **median**, **worst**, and one *random* frozen-val draw (so the selection is not cherry-picked).  Saved as `frozen_validation_viewer_gallery_*.png` — drop these straight into the deck.  

Also prints a LaTeX-ready table for the paper.

In [ ]:
import pandas as pd
from matplotlib.colors import ListedColormap

# --- fixed palette (same as §5) ---
LABEL_CMAP = ListedColormap(["black","#d73027","#fee08b","#1a9850"])  # 0 bg, 1 NCR, 2 ED, 3 ET
LABEL_NAMES = {0: "bg", 1: "NCR", 2: "ED", 3: "ET"}

def tumor_slices(seg, k=3):
    """k slices with largest tumor area, sorted."""
    sums = seg.sum(axis=(1,2))
    idx = np.argsort(sums)[::-1]
    idx = [i for i in idx if sums[i] > 0][:k]
    if len(idx) < k:
        mid = seg.shape[0]//2
        idx += [mid]* (k-len(idx))
    return sorted(idx)

def overlay_errors(seg_slice, pred_slice, flair_slice):
    """RGB error map: green agreement, red FP, blue FN per region (WT view)."""
    gt = (seg_slice>0); pr = (pred_slice>0)
    tp = gt & pr; fp = (~gt) & pr; fn = gt & (~pr)
    # base FLAIR gray
    base = plt.cm.gray((flair_slice - flair_slice.min())/(flair_slice.ptp()+1e-8))
    out = base[...,:3].copy()
    # tint
    out[tp] = out[tp]*0.5 + np.array([0.2,0.8,0.2]) * 0.5
    out[fp] = out[fp]*0.5 + np.array([0.9,0.2,0.2]) * 0.5
    out[fn] = out[fn]*0.5 + np.array([0.2,0.3,0.9]) * 0.5
    return out

df = pd.DataFrame(rows)
df["mean_dice"] = df[["dice_ET","dice_TC","dice_WT"]].mean(axis=1)
df_sorted = df.sort_values("mean_dice", ascending=False)
pick_ids = {
    "best": df_sorted.iloc[0]["case"],
    "median": df_sorted.iloc[len(df_sorted)//2]["case"],
    "worst": df_sorted.iloc[-1]["case"],
    "random": df.sample(1, random_state=17).iloc[0]["case"],
}
print("gallery picks:", pick_ids)
display(df_sorted[["case","dice_ET","dice_TC","dice_WT","mean_dice"]].head(8).style.format(precision=3).background_gradient(cmap="Greens"))
print("\nLaTeX row (ET / TC / WT, mean±std):")
for r in ("ET","TC","WT"):
    print(f"{r} & ${summary[r]['mean']:.3f} \\pm {summary[r]['std']:.3f}$ & {summary[r]['min']:.3f} & {summary[r]['max']:.3f} & {summary[r]['constant_mean']:.3f} \\\\")
print(f"\\midrule\nMean & ${df['mean_dice'].mean():.3f}$ & & & ")

In [ ]:
# --- render gallery: 4 cases × (3 slices) ---
for tag, cid in pick_ids.items():
    rec = all_recs[cid]
    vol, seg = load_case(rec)
    pred = sliding_window_logits(model, vol, CONFIG["patch"], CONFIG["stride"], device).argmax(0).astype(np.int16)
    gt_r, pd_r = regions(seg), regions(pred)
    scores = {r: dice_empty_rule(pd_r[r], gt_r[r]) for r in ("ET","TC","WT")}
    sls = tumor_slices(seg, k=3)
    fig, axes = plt.subplots(5, 3, figsize=(12, 14))
    fig.suptitle(f"{tag.upper()}  {cid}  ET {scores['ET']:.3f}  TC {scores['TC']:.3f}  WT {scores['WT']:.3f}  [{CONFIG['model'].upper()}]", fontsize=11)
    for col, si in enumerate(sls):
        # normalize FLAIR/T1c window for display
        flair = vol[3, si]; t1c = vol[1, si]
        for v in (flair, t1c):
            lo, hi = np.percentile(v[v>0] if (v>0).any() else v, [1, 99.5])
            v[:] = np.clip((v-lo)/(hi-lo+1e-8), 0, 1)  # in-place for compactness is fine for display copy
        axes[0,col].imshow(t1c, cmap="gray"); axes[0,col].set_title(f"T1c  z={si}" if col==1 else f"z={si}", fontsize=9); axes[0,col].axis("off")
        axes[1,col].imshow(flair, cmap="gray"); axes[1,col].set_title("FLAIR", fontsize=9); axes[1,col].axis("off")
        axes[2,col].imshow(seg[si], cmap=LABEL_CMAP, vmin=0, vmax=3); axes[2,col].set_title("GT", fontsize=9); axes[2,col].axis("off")
        axes[3,col].imshow(pred[si], cmap=LABEL_CMAP, vmin=0, vmax=3); axes[3,col].set_title("Pred", fontsize=9); axes[3,col].axis("off")
        axes[4,col].imshow(overlay_errors(seg[si], pred[si], flair)); axes[4,col].set_title("Error (G+agree R+FP B+FN)", fontsize=8); axes[4,col].axis("off")
    handles = [mpatches.Patch(color=LABEL_CMAP(i), label=f"{i} {LABEL_NAMES[i]}") for i in range(4)]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=9)
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    out = project_root / "notebooks" / f"frozen_validation_viewer_gallery_{tag}_{cid}.png"
    plt.savefig(out, dpi=180, bbox_inches="tight")
    print(f"saved {out}")
    plt.show()


## 8 — What H0 will change (and what it will not)
B1 is frozen exactly as above: `SharedStem → CNNEncoder → UNetDecoder`.  H0 imports those three classes *without modification* and inserts:

```
skips = encoder(stem(x))          # same 4 tensors, same shapes
bottleneck = skips[-1]            # 128×6³
bottleneck = vit(bottleneck)      # NEW — no shape change
skips[-1] = fusion(bottleneck, skips[-1])  # NEW — e.g. gated concat
logits = decoder(skips)           # same decoder
```

Budget, patch sampler, loss (Dice+0.5·CE), optimizer, and split are locked by `train_segmentation.py`.  That makes `B1 vs H0` a pure architecture test — the number the professor asked for.  The orange-shaded rows in the table are the Gate-2 line: if H0 does not beat the *frozen-val* B0/B1 WT mean printed in §6, it does not promote.

In [ ]:
# quick sanity: architecture param counts (should be close; H0 will add ViT params on top)
from scanvidence.models.cnn_shared import B1Segmentor
from scanvidence.models.backbone import SegResNetB0
for name, M in [("B0 (SegResNetB0)", SegResNetB0), ("B1 (cnn_shared)", B1Segmentor)]:
    m = M(in_channels=4, num_classes=4, widths=CONFIG["widths"])
    print(f"{name:18s}  {sum(p.numel() for p in m.parameters()):>9,} params")
print("\nB1 shares SharedStem/CNNEncoder/UNetDecoder with H0 — diff is only the future ViT+fusion.")

_All figures in this notebook are deterministic from `CONFIG['seed']` and the split hash. Re-running writes identical PNGs — paste the hash (`split_hash`) into the slide footer._